# Caso práctico: análisis de datos de la Copa Mundial con Apache Spark

**Asignatura:** 202681.2558 | Gestión de Datos y Tecnologías

**Integrantes:**

- Eduardo Garrido
- Luis Espinosa
- Mauricio Ortega
- Wilson Arévalo

Notebook desarrollado en **PySpark** para Google Colab.

El desarrollo sigue el orden de la guía:

1. Configuración del entorno.
2. Creación y unión de RDD.
3. Transformaciones sobre RDD.
4. Creación, integración y optimización de DataFrames.
5. Particionamiento.
6. Inspección del DataFrame.
7. Columnas calculadas.
8. Consultas con Spark SQL.
9. Transformaciones avanzadas.

> Ejecute las celdas en orden. Los cinco archivos de datos deben conservar sus nombres originales.


## Carga de los archivos

En Google Colab, ejecute la siguiente celda y seleccione:

- `jugadores.csv`
- `equipos.csv`
- `partidos.csv`
- `estadios.csv`
- `torneos.json`

In [ ]:
from google.colab import files
import os

ARCHIVOS_ESPERADOS = {
    "jugadores.csv",
    "equipos.csv",
    "partidos.csv",
    "estadios.csv",
    "torneos.json",
}

# Evita solicitar nuevamente los archivos si ya existen en /content.
faltantes = [nombre for nombre in ARCHIVOS_ESPERADOS
             if not os.path.exists(f"/content/{nombre}")]

if faltantes:
    print("Seleccione los archivos faltantes:", faltantes)
    files.upload()
else:
    print("Los cinco archivos ya están disponibles en /content.")

faltantes = [nombre for nombre in ARCHIVOS_ESPERADOS
             if not os.path.exists(f"/content/{nombre}")]

if faltantes:
    raise FileNotFoundError(
        "No se encontraron los siguientes archivos: " + ", ".join(faltantes)
    )

BASE_PATH = "/content"
print("Archivos validados correctamente.")

# Parte 1: configuración inicial del entorno

Se instala Java y PySpark, se definen las variables de entorno y se crea una
`SparkSession` llamada **MundialAnalysis**.

In [ ]:
# Instalación para Google Colab.
!apt-get update -qq
!apt-get install -y openjdk-17-jdk-headless -qq > /dev/null
!pip install -q pyspark

In [ ]:
import os
from pyspark.sql import SparkSession

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["SPARK_HOME"] = os.path.dirname(
    __import__("pyspark").__file__
)

spark = (
    SparkSession.builder
    .appName("MundialAnalysis")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

sc = spark.sparkContext
sc.setLogLevel("WARN")

print("Aplicación:", sc.appName)
print("Versión de Spark:", spark.version)
print("Paralelismo predeterminado:", sc.defaultParallelism)

# Parte 2: RDD — creación y unión

Los archivos se leen como texto, se elimina el encabezado y cada fila se
convierte a una tupla con tipos numéricos apropiados.

In [ ]:
import csv
from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    StringType,
)

RUTA_JUGADORES = f"{BASE_PATH}/jugadores.csv"

def parsear_jugador(linea):
    valores = next(csv.reader([linea]))
    return (
        int(valores[0]),   # jugador_id
        valores[1],        # nombre
        valores[2],        # apellido
        int(valores[3]),   # edad
        int(valores[4]),   # altura
        int(valores[5]),   # peso
        valores[6],        # posicion
        int(valores[7]),   # equipo_id
    )

def crear_rdd_jugadores(ruta):
    rdd_texto = sc.textFile(ruta, minPartitions=6)
    encabezado = rdd_texto.first()

    # Se fuerza explícitamente el resultado a seis particiones.
    return (
        rdd_texto
        .filter(lambda fila: fila != encabezado)
        .map(parsear_jugador)
        .repartition(6)
    )

# 2.1 y 2.2: dos fuentes simuladas con seis particiones cada una.
jugador1 = crear_rdd_jugadores(RUTA_JUGADORES)
jugador2 = crear_rdd_jugadores(RUTA_JUGADORES)

print("Particiones de jugador1:", jugador1.getNumPartitions())
print("Particiones de jugador2:", jugador2.getNumPartitions())
print("Primeros registros de jugador1:")
for fila in jugador1.take(5):
    print(fila)

In [ ]:
# 2.3: unión de ambos RDD.
jugadorTotal = jugador1.union(jugador2)

# 2.4: cantidad total de registros.
cantidad_total = jugadorTotal.count()

print("Particiones de jugadorTotal:", jugadorTotal.getNumPartitions())
print("Cantidad de registros en jugadorTotal:", cantidad_total)

In [ ]:
# 2.5: conversión del RDD unido a DataFrame mediante un esquema explícito.
esquema_jugadores = StructType([
    StructField("jugador_id", IntegerType(), False),
    StructField("nombre_jugador", StringType(), False),
    StructField("apellido", StringType(), False),
    StructField("edad", IntegerType(), False),
    StructField("altura", IntegerType(), False),
    StructField("peso", IntegerType(), False),
    StructField("posicion", StringType(), False),
    StructField("equipo_id", IntegerType(), False),
])

jugadores = spark.createDataFrame(jugadorTotal, esquema_jugadores)

jugadores.show(10, truncate=False)

In [ ]:
# 2.6: esquema y tipos de datos utilizables.
jugadores.printSchema()
print("Tipos de datos:", jugadores.dtypes)

# Parte 3: RDD — transformaciones

In [ ]:
# 3.1: jugadores mayores de 30 años.
MayorEdad = jugadorTotal.filter(lambda jugador: jugador[3] > 30)

print("Cantidad de jugadores mayores de 30 años:", MayorEdad.count())
for fila in MayorEdad.take(10):
    print(fila)

In [ ]:
# 3.2: jugadores cuya posición es Defensa.
Jugadores_Defensa = jugadorTotal.filter(
    lambda jugador: jugador[6].strip().lower() == "defensa"
)

print("Cantidad de defensas:", Jugadores_Defensa.count())
for fila in Jugadores_Defensa.take(10):
    print(fila)

In [ ]:
# 3.3: nombre y apellido convertidos a mayúsculas.
Jugadores_Mayusculas = jugadorTotal.map(
    lambda jugador: (
        jugador[0],
        jugador[1].upper(),
        jugador[2].upper(),
        jugador[3],
        jugador[4],
        jugador[5],
        jugador[6],
        jugador[7],
    )
)

for fila in Jugadores_Mayusculas.take(10):
    print(fila)

# Parte 4: DataFrames — creación, optimización e integración

Para evitar nombres ambiguos durante los `join`, algunas columnas descriptivas
se renombran. Después se construye una tabla de participaciones con una fila por
equipo y partido, diferenciando el rol local y visitante.

Finalmente, cada jugador se relaciona con los partidos disputados por su equipo.
Así, `mundial_completo` integra jugadores, equipos, partidos, estadios y torneos.

In [ ]:
from pyspark.sql import functions as F
from pyspark.storagelevel import StorageLevel

equipos = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{BASE_PATH}/equipos.csv")
    .withColumnRenamed("nombre", "nombre_equipo")
)

partidos = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{BASE_PATH}/partidos.csv")
)

estadios = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{BASE_PATH}/estadios.csv")
    .withColumnRenamed("nombre", "nombre_estadio")
    .withColumnRenamed("ciudad", "ciudad_estadio")
    .withColumnRenamed("pais", "pais_estadio")
)

torneos = (
    spark.read
    .option("multiline", True)
    .json(f"{BASE_PATH}/torneos.json")
    .withColumnRenamed("nombre", "nombre_torneo")
)

print("Equipos")
equipos.show(5, truncate=False)

print("Partidos")
partidos.show(5, truncate=False)

print("Estadios")
estadios.show(5, truncate=False)

print("Torneos")
torneos.show(5, truncate=False)

In [ ]:
# 4.2: persistencia y materialización de los DataFrames.
jugadores.persist(StorageLevel.MEMORY_AND_DISK)
equipos.cache()
partidos.cache()
estadios.cache()
torneos.cache()

for nombre, df in [
    ("jugadores", jugadores),
    ("equipos", equipos),
    ("partidos", partidos),
    ("estadios", estadios),
    ("torneos", torneos),
]:
    print(f"{nombre}: {df.count()} registros; almacenamiento = {df.storageLevel}")

In [ ]:
# Detalle de partido con equipos local y visitante, estadio y torneo.
equipo_local = equipos.select(
    F.col("equipo_id").alias("equipo_local_id"),
    F.col("nombre_equipo").alias("nombre_equipo_local"),
    F.col("confederacion").alias("confederacion_local"),
    F.col("ranking_fifa").alias("ranking_fifa_local"),
)

equipo_visitante = equipos.select(
    F.col("equipo_id").alias("equipo_visitante_id"),
    F.col("nombre_equipo").alias("nombre_equipo_visitante"),
    F.col("confederacion").alias("confederacion_visitante"),
    F.col("ranking_fifa").alias("ranking_fifa_visitante"),
)

partidos_detalle = (
    partidos
    .join(equipo_local, "equipo_local_id", "inner")
    .join(equipo_visitante, "equipo_visitante_id", "inner")
    .join(estadios, "estadio_id", "inner")
    .join(torneos, "torneo_id", "inner")
)

partidos_detalle.show(5, truncate=False)

In [ ]:
# Una fila por equipo participante en cada partido.
participaciones_local = partidos_detalle.select(
    "partido_id",
    "torneo_id",
    "nombre_torneo",
    "anio",
    "pais_sede",
    "campeon",
    "subcampeon",
    "estadio_id",
    "nombre_estadio",
    "ciudad_estadio",
    "pais_estadio",
    "capacidad",
    "fase",
    "goles_local",
    "goles_visitante",
    F.col("equipo_local_id").alias("equipo_id"),
    F.lit("Local").alias("condicion"),
    F.col("goles_local").alias("goles_favor"),
    F.col("goles_visitante").alias("goles_contra"),
    F.when(F.col("goles_local") > F.col("goles_visitante"), "Victoria")
     .when(F.col("goles_local") < F.col("goles_visitante"), "Derrota")
     .otherwise("Empate")
     .alias("resultado_equipo"),
)

participaciones_visitante = partidos_detalle.select(
    "partido_id",
    "torneo_id",
    "nombre_torneo",
    "anio",
    "pais_sede",
    "campeon",
    "subcampeon",
    "estadio_id",
    "nombre_estadio",
    "ciudad_estadio",
    "pais_estadio",
    "capacidad",
    "fase",
    "goles_local",
    "goles_visitante",
    F.col("equipo_visitante_id").alias("equipo_id"),
    F.lit("Visitante").alias("condicion"),
    F.col("goles_visitante").alias("goles_favor"),
    F.col("goles_local").alias("goles_contra"),
    F.when(F.col("goles_visitante") > F.col("goles_local"), "Victoria")
     .when(F.col("goles_visitante") < F.col("goles_local"), "Derrota")
     .otherwise("Empate")
     .alias("resultado_equipo"),
)

participaciones = participaciones_local.unionByName(
    participaciones_visitante
)

print("Participaciones esperadas: dos por partido")
print("Cantidad de participaciones:", participaciones.count())
participaciones.show(10, truncate=False)

In [ ]:
# 4.3: integración de los cinco conjuntos de datos.
jugadores_equipos = jugadores.join(equipos, "equipo_id", "inner")

mundial_completo = (
    jugadores_equipos
    .join(participaciones, "equipo_id", "inner")
)

print("Cantidad de filas del DataFrame integrado:", mundial_completo.count())
mundial_completo.show(10, truncate=False)

# Parte 5: paralelismo

In [ ]:
# 5.1: reparto del DataFrame integrado en cinco particiones.
mundial_completo = mundial_completo.repartition(5, "partido_id")

# 5.2: verificación.
print(
    "Número de particiones de mundial_completo:",
    mundial_completo.rdd.getNumPartitions()
)

# Parte 6: inspección del DataFrame

In [ ]:
# 6.1: cantidad de filas, tipos de datos y esquema.
print("Cantidad de filas:", mundial_completo.count())
print("Tipos de datos:", mundial_completo.dtypes)
mundial_completo.printSchema()

# Parte 7: columnas calculadas

In [ ]:
# 7.1: IMC = peso / (altura en metros)^2.
mundial_completo = mundial_completo.withColumn(
    "IMC",
    F.round(
        F.col("peso") / F.pow(F.col("altura") / F.lit(100.0), 2),
        2,
    ),
)

# 7.2: categoría de edad.
mundial_completo = mundial_completo.withColumn(
    "Categoria_Edad",
    F.when(F.col("edad") < 25, "Joven")
     .when(F.col("edad").between(25, 32), "Experimentado")
     .otherwise("Veterano"),
)

# 7.3: resultado desde la perspectiva del equipo local.
mundial_completo = mundial_completo.withColumn(
    "Resultado_Partido",
    F.when(F.col("goles_local") > F.col("goles_visitante"), "Victoria Local")
     .when(F.col("goles_local") < F.col("goles_visitante"), "Victoria Visitante")
     .otherwise("Empate"),
)

mundial_completo.select(
    "jugador_id",
    "nombre_jugador",
    "apellido",
    "edad",
    "peso",
    "altura",
    "IMC",
    "Categoria_Edad",
    "partido_id",
    "goles_local",
    "goles_visitante",
    "Resultado_Partido",
).show(20, truncate=False)

# Parte 8: agregaciones con Spark SQL

Todas las respuestas de esta sección se obtienen con `spark.sql()`.

El DataFrame integrado contiene repeticiones naturales: un jugador aparece en
cada partido de su equipo y, además, `jugadorTotal` contiene las dos fuentes
unidas. Por ello, las consultas utilizan `DISTINCT` cuando corresponde para no
duplicar jugadores, partidos, estadios o participaciones.

In [ ]:
# Registro de la vista temporal solicitada.
mundial_completo.createOrReplaceTempView("Mundial")

print("Vista temporal 'Mundial' registrada.")

## 8.1 ¿Cuántos jugadores hay por equipo?

In [ ]:
consulta_8_1 = spark.sql('''
    SELECT
        nombre_equipo,
        COUNT(DISTINCT jugador_id) AS cantidad_jugadores
    FROM Mundial
    GROUP BY nombre_equipo
    ORDER BY cantidad_jugadores DESC, nombre_equipo
''')

consulta_8_1.show(truncate=False)

## 8.2 Edad promedio, mínima y máxima, agrupada por posición

In [ ]:
consulta_8_2 = spark.sql('''
    WITH jugadores_unicos AS (
        SELECT DISTINCT jugador_id, posicion, edad
        FROM Mundial
    )
    SELECT
        posicion,
        ROUND(AVG(edad), 2) AS edad_promedio,
        MIN(edad) AS edad_minima,
        MAX(edad) AS edad_maxima
    FROM jugadores_unicos
    GROUP BY posicion
    ORDER BY posicion
''')

consulta_8_2.show(truncate=False)

## 8.3 Altura promedio, mínima y máxima, agrupada por confederación

In [ ]:
consulta_8_3 = spark.sql('''
    WITH jugadores_unicos AS (
        SELECT DISTINCT jugador_id, confederacion, altura
        FROM Mundial
    )
    SELECT
        confederacion,
        ROUND(AVG(altura), 2) AS altura_promedio,
        MIN(altura) AS altura_minima,
        MAX(altura) AS altura_maxima
    FROM jugadores_unicos
    GROUP BY confederacion
    ORDER BY confederacion
''')

consulta_8_3.show(truncate=False)

## 8.4 ¿Cuántos partidos se jugaron en cada fase?

In [ ]:
consulta_8_4 = spark.sql('''
    SELECT
        fase,
        COUNT(DISTINCT partido_id) AS cantidad_partidos
    FROM Mundial
    GROUP BY fase
    ORDER BY cantidad_partidos DESC, fase
''')

consulta_8_4.show(truncate=False)

## 8.5 Total de goles anotados por cada equipo

In [ ]:
consulta_8_5 = spark.sql('''
    WITH participaciones_unicas AS (
        SELECT DISTINCT
            partido_id,
            equipo_id,
            nombre_equipo,
            goles_favor
        FROM Mundial
    )
    SELECT
        nombre_equipo,
        SUM(goles_favor) AS total_goles
    FROM participaciones_unicas
    GROUP BY nombre_equipo
    ORDER BY total_goles DESC, nombre_equipo
''')

consulta_8_5.show(truncate=False)

## 8.6 Promedio de goles por partido en cada torneo

In [ ]:
consulta_8_6 = spark.sql('''
    WITH partidos_unicos AS (
        SELECT DISTINCT
            partido_id,
            nombre_torneo,
            goles_local,
            goles_visitante
        FROM Mundial
    )
    SELECT
        nombre_torneo,
        ROUND(AVG(goles_local + goles_visitante), 2)
            AS promedio_goles_por_partido
    FROM partidos_unicos
    GROUP BY nombre_torneo
    ORDER BY nombre_torneo
''')

consulta_8_6.show(truncate=False)

## 8.7 Capacidad promedio de los estadios agrupados por país

In [ ]:
consulta_8_7 = spark.sql('''
    WITH estadios_unicos AS (
        SELECT DISTINCT
            estadio_id,
            pais_estadio,
            capacidad
        FROM Mundial
    )
    SELECT
        pais_estadio,
        ROUND(AVG(capacidad), 2) AS capacidad_promedio
    FROM estadios_unicos
    GROUP BY pais_estadio
    ORDER BY capacidad_promedio DESC, pais_estadio
''')

consulta_8_7.show(truncate=False)

## 8.8 Clasificación del IMC mediante `CASE WHEN`

In [ ]:
consulta_8_8 = spark.sql('''
    WITH jugadores_unicos AS (
        SELECT DISTINCT
            jugador_id,
            nombre_jugador,
            apellido,
            IMC
        FROM Mundial
    )
    SELECT
        jugador_id,
        nombre_jugador,
        apellido,
        IMC,
        CASE
            WHEN IMC < 20 THEN 'Bajo peso'
            WHEN IMC BETWEEN 20 AND 25 THEN 'Normal'
            ELSE 'Sobrepeso'
        END AS Clasificacion_IMC
    FROM jugadores_unicos
    ORDER BY jugador_id
''')

consulta_8_8.show(80, truncate=False)

## 8.9 Número de victorias, derrotas y empates de cada equipo

In [ ]:
consulta_8_9 = spark.sql('''
    WITH participaciones_unicas AS (
        SELECT DISTINCT
            partido_id,
            equipo_id,
            nombre_equipo,
            resultado_equipo
        FROM Mundial
    )
    SELECT
        nombre_equipo,
        SUM(CASE WHEN resultado_equipo = 'Victoria' THEN 1 ELSE 0 END)
            AS victorias,
        SUM(CASE WHEN resultado_equipo = 'Derrota' THEN 1 ELSE 0 END)
            AS derrotas,
        SUM(CASE WHEN resultado_equipo = 'Empate' THEN 1 ELSE 0 END)
            AS empates
    FROM participaciones_unicas
    GROUP BY nombre_equipo
    ORDER BY victorias DESC, nombre_equipo
''')

consulta_8_9.show(truncate=False)

## 8.10 Equipos con edad promedio superior a 28 años usando `HAVING`

In [ ]:
consulta_8_10 = spark.sql('''
    WITH jugadores_unicos AS (
        SELECT DISTINCT
            jugador_id,
            nombre_equipo,
            edad
        FROM Mundial
    )
    SELECT
        nombre_equipo,
        ROUND(AVG(edad), 2) AS edad_promedio
    FROM jugadores_unicos
    GROUP BY nombre_equipo
    HAVING AVG(edad) > 28
    ORDER BY edad_promedio DESC, nombre_equipo
''')

consulta_8_10.show(truncate=False)

# Parte 9: transformaciones avanzadas

In [ ]:
# 9.1: partidos de la fase Final.
partidos_final = (
    mundial_completo
    .filter(F.col("fase") == "Final")
    .select(
        "partido_id",
        "nombre_torneo",
        "nombre_estadio",
        "goles_local",
        "goles_visitante",
        "Resultado_Partido",
    )
    .dropDuplicates(["partido_id"])
)

partidos_final.show(truncate=False)

In [ ]:
# 9.2: selección de las columnas solicitadas.
jugadores_seleccionados = (
    mundial_completo
    .select(
        "jugador_id",
        "nombre_jugador",
        "apellido",
        "posicion",
        "nombre_equipo",
    )
    .dropDuplicates(["jugador_id"])
)

jugadores_seleccionados.show(80, truncate=False)

In [ ]:
# 9.3: jugadores ordenados por edad de mayor a menor.
jugadores_por_edad = (
    mundial_completo
    .select(
        "jugador_id",
        "nombre_jugador",
        "apellido",
        "edad",
        "nombre_equipo",
    )
    .dropDuplicates(["jugador_id"])
    .orderBy(F.col("edad").desc(), F.col("apellido"), F.col("nombre_jugador"))
)

jugadores_por_edad.show(80, truncate=False)

In [ ]:
# 9.4: los diez jugadores más altos.
jugadores_mas_altos = (
    mundial_completo
    .select(
        "jugador_id",
        "nombre_jugador",
        "apellido",
        "altura",
        "posicion",
        "nombre_equipo",
    )
    .dropDuplicates(["jugador_id"])
    .orderBy(F.col("altura").desc(), F.col("apellido"), F.col("nombre_jugador"))
    .limit(10)
)

jugadores_mas_altos.show(10, truncate=False)

# Validaciones finales

Estas verificaciones ayudan a detectar problemas comunes antes de comparar el
trabajo con el solucionario.

In [ ]:
validaciones = {
    "RDD jugador1 tiene 6 particiones": jugador1.getNumPartitions() == 6,
    "RDD jugador2 tiene 6 particiones": jugador2.getNumPartitions() == 6,
    "La unión contiene el doble de registros que una fuente":
        jugadorTotal.count() == 2 * jugador1.count(),
    "mundial_completo tiene 5 particiones":
        mundial_completo.rdd.getNumPartitions() == 5,
    "La vista temporal Mundial existe":
        spark.catalog.tableExists("Mundial"),
    "IMC fue creada": "IMC" in mundial_completo.columns,
    "Categoria_Edad fue creada":
        "Categoria_Edad" in mundial_completo.columns,
    "Resultado_Partido fue creada":
        "Resultado_Partido" in mundial_completo.columns,
}

for descripcion, resultado in validaciones.items():
    estado = "OK" if resultado else "REVISAR"
    print(f"[{estado}] {descripcion}")

In [ ]:
# Liberación opcional de recursos al terminar.
# jugadores.unpersist()
# equipos.unpersist()
# partidos.unpersist()
# estadios.unpersist()
# torneos.unpersist()
# spark.stop()